<a href="https://colab.research.google.com/github/jen257/cs4207-coding-challenge/blob/main/CS4207_Group4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Setup & Config

In [8]:
import os
import sys
import gc
import time
import platform
import multiprocessing as mp
from pathlib import Path
from tempfile import TemporaryDirectory
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from joblib import wrap_non_picklable_objects

START_TIME = time.perf_counter()

# Automatically switch between Rockhill server and local Windows VS Code
if os.path.exists("/home/cs4207/data"):
    DATA_DIR = "/home/cs4207/data"
else:
    DATA_DIR = None
    for base in [Path(r"C:\CS4207"), Path.home() / "Downloads", Path.home() / "Desktop", Path(".")]:
        if base.exists():
            matches = list(base.rglob("yellow_tripdata_2026-01.parquet"))
            if matches:
                DATA_DIR = str(matches[0].parent)
                break
    if DATA_DIR is None:
        raise FileNotFoundError("Could not find yellow_tripdata_2026-01.parquet.")

TRAIN_FILES = [
    os.path.join(DATA_DIR, "yellow_tripdata_2026-01.parquet"),
    os.path.join(DATA_DIR, "yellow_tripdata_2026-02.parquet"),
]
VAL_FILE = os.path.join(DATA_DIR, "yellow_tripdata_2026-03.parquet")
TEST_FILE = os.path.join(DATA_DIR, "yellow_tripdata_2026-04.parquet")
ALL_FILES = TRAIN_FILES + [VAL_FILE, TEST_FILE]

# Global settings
SEED = 42
WORKERS = [1, 2, 4]
REPEATS = 3
np.random.seed(SEED)

# Environment & hardware info
print("Platform:", platform.platform())
print("Python:", sys.version.split()[0])
print("CPU cores:", os.cpu_count())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)
print("sklearn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
print("Data dir:", DATA_DIR)

for f in ALL_FILES:
    status = "FOUND" if os.path.exists(f) else "MISSING"
    print(f"  [{status}] {os.path.basename(f)}")

Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Python: 3.13.18
CPU cores: 2
numpy: 2.4.6
pandas: 2.2.3
pyarrow: 23.0.1
sklearn: 1.6.1
matplotlib: 3.10.0
Data dir: drive/MyDrive/CS4207
  [FOUND] yellow_tripdata_2026-01.parquet
  [FOUND] yellow_tripdata_2026-02.parquet
  [FOUND] yellow_tripdata_2026-03.parquet
  [FOUND] yellow_tripdata_2026-04.parquet


Cell 2 — Data Cleaning & Data Parallelism

In [9]:
CLEAN_WORKERS = 2
MAX_MILES = 100
MIN_MINUTES, MAX_MINUTES = 1, 180
USE_COLS = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
    "trip_distance", "fare_amount", "PULocationID", "DOLocationID",
    "VendorID", "RatecodeID", "payment_type",
]
CATEGORY_COLS = ["PULocationID", "DOLocationID", "RatecodeID"]
FEATURE_COLUMNS = [col for col in USE_COLS if col != "payment_type"]


def _clean_shard(job):
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq

    path, month, group, output = job
    # Column projection excludes tip_amount and total_amount (payment leakage).
    df = pq.ParquetFile(path).read_row_group(group, columns=USE_COLS, use_threads=False).to_pandas()
    active = np.ones(len(df), dtype=bool)
    removed = {}

    def keep(rule, condition):
        passed = condition.fillna(False).to_numpy(dtype=bool)
        removed[rule] = int(np.count_nonzero(active & ~passed))
        active[:] &= passed

    pickup = df["tpep_pickup_datetime"]
    duration = df["tpep_dropoff_datetime"] - pickup
    fare, miles, pay = df["fare_amount"], df["trip_distance"], df["payment_type"]
    keep("Pickup in file month", pickup.ge(month.start_time) & pickup.lt((month + 1).start_time))
    keep("Exclude Flex Fare (0)", ~pay.isin([0]))
    keep("Keep card / cash (1, 2)", pay.isin([1, 2]))
    keep("Finite fare > 0", np.isfinite(fare) & fare.gt(0))
    keep("Finite distance > 0", np.isfinite(miles) & miles.gt(0))
    keep(f"Distance < {MAX_MILES} miles", miles.lt(MAX_MILES))
    keep(f"Duration >= {MIN_MINUTES} minute", duration.ge(pd.Timedelta(minutes=MIN_MINUTES)))
    keep(f"Duration <= {MAX_MINUTES} minutes", duration.le(pd.Timedelta(minutes=MAX_MINUTES)))
    keep("Complete required fields", df[USE_COLS].notna().all(axis=1))

    clean = df.loc[active, USE_COLS].copy()
    clean.index.name = "source_row_in_group"
    clean.to_parquet(output, engine="pyarrow", index=True)
    return path, group, len(df), removed, output


def _run_shards(files, n_workers, folder):
    folder.mkdir()
    jobs = []
    for path in map(Path, files):
        month = pd.Period(path.stem.rsplit("_", 1)[-1], freq="M")
        metadata = pq.ParquetFile(path).metadata
        for group in range(metadata.num_row_groups):
            jobs.append((path, month, group, folder / f"shard-{len(jobs)}.parquet"))
    if n_workers == 1:
        return [_clean_shard(job) for job in jobs]
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context("spawn")) as pool:
        return list(pool.map(wrap_non_picklable_objects(_clean_shard), jobs))


def _assemble(frames):
    merged = pd.concat(frames, ignore_index=True)
    merged[CATEGORY_COLS] = merged[CATEGORY_COLS].astype("category")
    return merged


def clean_parallel(files, n_workers=CLEAN_WORKERS):
    """Same ordered files, row groups and merge for 1/2/4 workers."""
    with TemporaryDirectory() as staging:
        shards = _run_shards(files, n_workers, Path(staging) / "shards")
        return _assemble([pd.read_parquet(result[-1]) for result in shards])


def clean_sequential(files):
    return clean_parallel(files, n_workers=1)


def clean_one_file(file_path):
    return clean_sequential([file_path])


def _verify_month(file_path, split):
    month = pd.Period(Path(file_path).stem.rsplit("_", 1)[-1], freq="M")
    frames, counts = [], {}
    footer = pq.ParquetFile(file_path).metadata
    with TemporaryDirectory() as staging:
        folder = Path(staging)
        serial = _run_shards([file_path], 1, folder / "seq")
        parallel = _run_shards([file_path], CLEAN_WORKERS, folder / "par")
        assert len(serial) == len(parallel) == footer.num_row_groups
        for group, (left, right) in enumerate(zip(serial, parallel)):
            assert left[:4] == right[:4] and right[1] == group
            assert right[2] == footer.row_group(group).num_rows
            expected, actual = pd.read_parquet(left[-1]), pd.read_parquet(right[-1])
            pd.testing.assert_frame_equal(expected, actual, check_exact=True, check_like=False)
            assert len(actual) + sum(right[3].values()) == right[2]
            frames.append(actual)
            for rule, removed in right[3].items():
                counts[rule] = counts.get(rule, 0) + removed

    cleaned = pd.concat(frames, ignore_index=True)
    pickup = cleaned["tpep_pickup_datetime"]
    assert pickup.ge(month.start_time).all() and pickup.lt((month + 1).start_time).all()
    assert cleaned["payment_type"].isin([1, 2]).all()
    rows, remaining = [], footer.num_rows
    for rule, removed in counts.items():
        rows.append((str(month), split, rule, remaining, removed, remaining - removed))
        remaining -= removed
    assert remaining == len(cleaned)
    summary = (str(month), split, footer.num_rows, remaining, footer.num_row_groups, "PASS")
    return cleaned, rows, summary


def _prepare_partitions():
    audits, summaries, partitions = [], [], []
    for split, files in [("train", TRAIN_FILES), ("validation", [VAL_FILE]), ("test", [TEST_FILE])]:
        frames = []
        for path in files:
            frame, rows, summary = _verify_month(path, split)
            frames.append(frame)
            audits.extend(rows)
            summaries.append(summary)
        partitions.append(_assemble(frames))
    stats = pd.DataFrame(audits, columns=["Month", "Split", "Rule", "Before", "Removed", "After"])
    summary = pd.DataFrame(summaries, columns=["Month", "Split", "Input", "Retained", "Row groups", "Equality"])
    return partitions, stats, summary


(train_df, val_df, test_df), cleaning_stats, monthly_summary = _prepare_partitions()
y_train, y_val, y_test = [
    df["payment_type"].eq(1).astype("int8").to_numpy()
    for df in (train_df, val_df, test_df)
]
pickup_col = "tpep_pickup_datetime"
assert train_df[pickup_col].max() < val_df[pickup_col].min()
assert val_df[pickup_col].max() < test_df[pickup_col].min()

print(cleaning_stats.to_string(index=False))
print(monthly_summary.to_string(index=False))
print(f"Split sizes | train {len(y_train):,} | validation {len(y_val):,} | test {len(y_test):,}")
print(f"PASS | sequential vs {CLEAN_WORKERS} workers: exact rows, order, types, values and per-rule counts.")
print("PASS | chronological splits and aligned binary labels (credit card=1, cash=0).")

  Month      Split                     Rule  Before  Removed   After
2026-01      train     Pickup in file month 3724889        7 3724882
2026-01      train    Exclude Flex Fare (0) 3724882  1088058 2636824
2026-01      train  Keep card / cash (1, 2) 2636824    73039 2563785
2026-01      train          Finite fare > 0 2563785    10130 2553655
2026-01      train      Finite distance > 0 2553655    24828 2528827
2026-01      train     Distance < 100 miles 2528827       36 2528791
2026-01      train     Duration >= 1 minute 2528791    51226 2477565
2026-01      train  Duration <= 180 minutes 2477565     1508 2476057
2026-01      train Complete required fields 2476057        0 2476057
2026-02      train     Pickup in file month 3399866       16 3399850
2026-02      train    Exclude Flex Fare (0) 3399850  1023317 2376533
2026-02      train  Keep card / cash (1, 2) 2376533    51272 2325261
2026-02      train          Finite fare > 0 2325261     7500 2317761
2026-02      train      Finite dis

Cell 3 — Task Parallelism A: Feature Engineering

In [10]:
def get_duration(df):
    """Task 1: Trip duration in minutes and log-duration."""
    import numpy as np
    import pandas as pd
    pickup = pd.to_datetime(df["tpep_pickup_datetime"])
    dropoff = pd.to_datetime(df["tpep_dropoff_datetime"])
    duration_min = (dropoff - pickup).dt.total_seconds() / 60.0
    log_duration = np.log1p(duration_min.clip(lower=0))
    return pd.DataFrame({
        "duration_min": duration_min,
        "log_duration": log_duration
    })

def get_time_features(df):
    """Task 2: Pickup hour (sin/cos), weekend, rush hour, night."""
    import numpy as np
    import pandas as pd
    pickup = pd.to_datetime(df["tpep_pickup_datetime"])
    hour = pickup.dt.hour
    dayofweek = pickup.dt.dayofweek

    hour_sin = np.sin(2 * np.pi * hour / 24.0)
    hour_cos = np.cos(2 * np.pi * hour / 24.0)

    is_weekend = dayofweek.isin([5, 6]).astype(float)
    is_rush_hour = ((hour.between(7, 9) | hour.between(16, 19)) & ~dayofweek.isin([5, 6])).astype(float)
    is_night = ((hour >= 20) | (hour < 6)).astype(float)

    return pd.DataFrame({
        "hour_sin": hour_sin,
        "hour_cos": hour_cos,
        "is_weekend": is_weekend,
        "is_rush_hour": is_rush_hour,
        "is_night": is_night
    })

def get_location_features(df, top_zones):
    """Task 3: Airport flags and top-zone one-hot encoding."""
    import pandas as pd
    pu = df["PULocationID"]
    do = df["DOLocationID"]

    features = {
        "is_jfk": ((pu == 132) | (do == 132)).astype(float),
        "is_lga": ((pu == 138) | (do == 138)).astype(float),
        "is_ewr": ((pu == 1) | (do == 1)).astype(float),
    }

    for zone in top_zones:
        features[f"pu_zone_{zone}"] = (pu == zone).astype(float)

    return pd.DataFrame(features)

def build_features_sequential(df, top_zones):
    """Sequential baseline for feature engineering."""
    d_feat = get_duration(df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]])
    t_feat = get_time_features(df[["tpep_pickup_datetime"]])
    l_feat = get_location_features(df[["PULocationID", "DOLocationID"]], top_zones)
    return pd.concat([
        df[["trip_distance", "fare_amount", "passenger_count"]].reset_index(drop=True),
        d_feat.reset_index(drop=True),
        t_feat.reset_index(drop=True),
        l_feat.reset_index(drop=True)
    ], axis=1)

def build_features_parallel(df, top_zones):
    """Run the 3 feature tasks concurrently (Windows & Linux safe)."""
    with ProcessPoolExecutor(max_workers=3, mp_context=mp.get_context("spawn")) as pool:
        f1 = pool.submit(wrap_non_picklable_objects(get_duration), df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]])
        f2 = pool.submit(wrap_non_picklable_objects(get_time_features), df[["tpep_pickup_datetime"]])
        f3 = pool.submit(wrap_non_picklable_objects(get_location_features), df[["PULocationID", "DOLocationID"]], top_zones)

        X = pd.concat([
            df[["trip_distance", "fare_amount", "passenger_count"]].reset_index(drop=True),
            f1.result().reset_index(drop=True),
            f2.result().reset_index(drop=True),
            f3.result().reset_index(drop=True)
        ], axis=1)
    return X

# Find top 15 pickup zones from cleaned train_df only (prevents leakage)
top_15_zones = train_df["PULocationID"].value_counts().head(15).index.tolist()

# Verify sequential and parallel feature builders produce identical outputs
sample_df = val_df.head(5000)
pd.testing.assert_frame_equal(
    build_features_sequential(sample_df, top_15_zones),
    build_features_parallel(sample_df, top_15_zones)
)
print("PASS | sequential vs parallel feature extraction match exactly.")

print("Building train, validation, and test features (float32)...")
X_train = build_features_parallel(train_df, top_15_zones).to_numpy(dtype=np.float32)
X_val = build_features_parallel(val_df, top_15_zones).to_numpy(dtype=np.float32)
X_test = build_features_parallel(test_df, top_15_zones).to_numpy(dtype=np.float32)

# Verify row alignment with labels from Cell 2
assert len(X_train) == len(y_train), f"Row mismatch: X_train={len(X_train)}, y_train={len(y_train)}"
assert len(X_val) == len(y_val), f"Row mismatch: X_val={len(X_val)}, y_val={len(y_val)}"
assert len(X_test) == len(y_test), f"Row mismatch: X_test={len(X_test)}, y_test={len(y_test)}"

# Free raw DataFrames to keep RAM low
del train_df, val_df, test_df
gc.collect()

print("Feature shapes:", X_train.shape, X_val.shape, X_test.shape)

PASS | sequential vs parallel feature extraction match exactly.
Building train, validation, and test features (float32)...
Feature shapes: (4723849, 28) (2869656, 28) (2905251, 28)


Cell 4 — Baseline Model & Task Parallelism B: Hyperparameter Search

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Ensure finite float32 values and scale once (fit on train only to prevent leakage)
X_train = np.clip(np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0), -1e6, 1e6)
X_val   = np.clip(np.nan_to_num(X_val,   nan=0.0, posinf=0.0, neginf=0.0), -1e6, 1e6)
X_test  = np.clip(np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0), -1e6, 1e6)

scaler = StandardScaler()
X_train = np.nan_to_num(scaler.fit_transform(X_train), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_val   = np.nan_to_num(scaler.transform(X_val),       nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test  = np.nan_to_num(scaler.transform(X_test),      nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
gc.collect()

# 1. Sequential Baseline
t0 = time.perf_counter()
baseline_model = LogisticRegression(max_iter=200, random_state=SEED)
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_val)
baseline_time = time.perf_counter() - t0

print("=== Sequential Baseline ===")
print(f"Time: {baseline_time:.2f}s | Val Accuracy: {accuracy_score(y_val, baseline_preds):.4f} | Val F1: {f1_score(y_val, baseline_preds):.4f}")

# 2. Task Parallelism: Parallel Hyperparameter Search
C_VALUES = [0.01, 0.1, 1.0, 10.0]

def train_one_model(c_val):
    """Train one model with a given C value and return its validation F1."""
    model = LogisticRegression(C=c_val, max_iter=200, random_state=SEED)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    score = f1_score(y_val, preds)
    return (c_val, score)

def run_tuning_parallel(c_list, n_workers=4):
    if n_workers == 1:
        return [train_one_model(c) for c in c_list]
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        results = list(pool.map(train_one_model, c_list))
    return results

print("\n=== Parallel Hyperparameter Search ===")
results = run_tuning_parallel(C_VALUES, n_workers=4)
for c, score in results:
    print(f"C={c} -> Validation F1={score:.4f}")

best_c, best_score = max(results, key=lambda x: x[1])
print(f"\nBest C: {best_c} | Best Validation F1: {best_score:.4f}")

# 3. Evaluate best model on test set
best_model = LogisticRegression(C=best_c, max_iter=200, random_state=SEED)
best_model.fit(X_train, y_train)
test_preds = best_model.predict(X_test)

print(f"\nTest Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test F1: {f1_score(y_test, test_preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=["Cash (0)", "Card (1)"]))

=== Sequential Baseline ===
Time: 14.39s | Val Accuracy: 0.8852 | Val F1: 0.9391

=== Parallel Hyperparameter Search ===
C=0.01 -> Validation F1=0.9391
C=0.1 -> Validation F1=0.9391
C=1.0 -> Validation F1=0.9391
C=10.0 -> Validation F1=0.9391

Best C: 0.01 | Best Validation F1: 0.9391

Test Accuracy: 0.8834
Test F1: 0.9380

Classification Report:
              precision    recall  f1-score   support

    Cash (0)       0.54      0.01      0.01    339061
    Card (1)       0.88      1.00      0.94   2566190

    accuracy                           0.88   2905251
   macro avg       0.71      0.50      0.48   2905251
weighted avg       0.84      0.88      0.83   2905251



Cell 5 — Benchmarking & Speedup Chart

In [ ]:
avg_times = []
outputs_by_w = {}

print("=== Starting Performance Evaluation ===")
for w in WORKERS:
    trial_times = []
    for trial in range(REPEATS):
        start_time = time.perf_counter()
        res = run_tuning_parallel(C_VALUES, n_workers=w)
        end_time = time.perf_counter()
        trial_times.append(end_time - start_time)
    outputs_by_w[w] = res
    avg_time = np.mean(trial_times)
    avg_times.append(avg_time)
    print(f"Workers: {w} | Trials: {[round(t, 4) for t in trial_times]} | Avg Time: {avg_time:.4f}s")

# Verify sequential (1 worker) and parallel (2, 4 workers) outputs match
base_scores = [score for _, score in outputs_by_w[1]]
for w in WORKERS[1:]:
    par_scores = [score for _, score in outputs_by_w[w]]
    assert np.allclose(base_scores, par_scores, atol=1e-6), f"Mismatch between 1 and {w} workers!"
print("PASS | Output equivalence verified across 1, 2, and 4 workers.")

# Calculate Speedup and Parallel Efficiency
t1 = avg_times[0]
speedups = [t1 / t for t in avg_times]
efficiencies = [s / w for s, w in zip(speedups, WORKERS)]

# Plot Speedup vs Workers
plt.figure(figsize=(8, 5))
plt.plot(WORKERS, speedups, marker="o", linewidth=2, label="Actual Speedup")
plt.plot(WORKERS, WORKERS, marker="x", linestyle="--", linewidth=2, label="Ideal Speedup (Linear)")
plt.title("Speedup vs. Number of Workers (Hyperparameter Tuning)")
plt.xlabel("Number of Workers")
plt.ylabel("Speedup ($T_1 / T_N$)")
plt.xticks(WORKERS)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("speedup_chart.png", dpi=300)
plt.show()

print("\n=== Summary Table ===")
print(f"{'Workers':<10} | {'Avg Time (s)':<14} | {'Speedup':<10} | {'Efficiency':<12}")
print("-" * 54)
for w, t, s, e in zip(WORKERS, avg_times, speedups, efficiencies):
    print(f"{w:<10} | {t:<14.4f} | {s:<9.2f}x | {e:<11.1%}")

=== Starting Performance Evaluation ===


### Amdahl's Law & Overhead Analysis
Actual speedup falls below the ideal linear curve because of **Amdahl's Law**:

$$S(N) = \frac{1}{(1 - p) + \frac{p}{N}}$$

The serial fraction $(1 - p)$ (task submission, scheduling, and result collection) limits maximum scaling. Practical overheads—including thread scheduling, memory bandwidth contention when multiple workers read `X_train` concurrently, BLAS/OpenMP thread contention inside scikit-learn, and synchronisation waiting for the slowest `C` value to converge—further reduce parallel efficiency as worker count increases.

Cell 6 — Non-Determinism & Fixes

In [12]:
from sklearn.linear_model import SGDClassifier

# Self-contained helper (works on both Windows spawn and Rockhill Linux)
def _read_first_fares(file_path):
    import os
    import pandas as pd
    df = pd.read_parquet(file_path, columns=["fare_amount", "payment_type"])
    df = df[(df["payment_type"].isin([1, 2])) & (df["fare_amount"] > 0)]
    return os.path.basename(file_path), df["fare_amount"].head(5).tolist()

# --- Source 1: Unordered worker completion (as_completed vs pool.map) ---
def run_unordered(files, n_workers=4):
    order, fares = [], []
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context("spawn")) as pool:
        futures = [pool.submit(wrap_non_picklable_objects(_read_first_fares), f) for f in files]
        for fut in as_completed(futures):
            name, first_fares = fut.result()
            order.append(name)
            fares.extend(first_fares)
    return order, fares[:5]

def run_ordered(files, n_workers=4):
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context("spawn")) as pool:
        results = list(pool.map(wrap_non_picklable_objects(_read_first_fares), files))
    order = [name for name, _ in results]
    fares = [val for _, first_fares in results for val in first_fares]
    return order, fares[:5]

# Before fix: as_completed() collects shards in whichever order workers finish
order1, u1 = run_unordered(ALL_FILES, n_workers=4)
order2, u2 = run_unordered(ALL_FILES, n_workers=4)
print("Source 1 (Before fix - as_completed):")
print("  Run 1 finish order:", order1, "| first 5 fares:", u1)
print("  Run 2 finish order:", order2, "| first 5 fares:", u2)

# After fix: pool.map() preserves fixed shard order
f_order1, f1 = run_ordered(ALL_FILES, n_workers=4)
f_order2, f2 = run_ordered(ALL_FILES, n_workers=4)
print("\nSource 1 (After fix - pool.map):")
print("  Run 1 order:", f_order1, "| first 5 fares:", f1)
print("  Run 2 order:", f_order2, "| first 5 fares:", f2)
assert f_order1 == f_order2 and f1 == f2
print("  PASS: Fixed shard-order runs match identically.")

# --- Source 2: Unseeded randomness (no random_state vs fixed SEED) ---
X_sub = X_train[:50000]
y_sub = y_train[:50000]

# Before fix: no random_state (tol=None runs 5 stochastic epochs without warning)
m1 = SGDClassifier(loss="log_loss", max_iter=5, tol=None)
m2 = SGDClassifier(loss="log_loss", max_iter=5, tol=None)
m1.fit(X_sub, y_sub)
m2.fit(X_sub, y_sub)
s1 = f1_score(y_val, m1.predict(X_val))
s2 = f1_score(y_val, m2.predict(X_val))
print(f"\nSource 2 (Before fix - unseeded SGD): Run 1 F1={s1:.6f} | Run 2 F1={s2:.6f}")

# After fix: fixed random_state=SEED
mf1 = SGDClassifier(loss="log_loss", max_iter=5, tol=None, random_state=SEED)
mf2 = SGDClassifier(loss="log_loss", max_iter=5, tol=None, random_state=SEED)
mf1.fit(X_sub, y_sub)
mf2.fit(X_sub, y_sub)
sf1 = f1_score(y_val, mf1.predict(X_val))
sf2 = f1_score(y_val, mf2.predict(X_val))
print(f"Source 2 (After fix - SEED={SEED}) : Run 1 F1={sf1:.6f} | Run 2 F1={sf2:.6f}")
assert sf1 == sf2
print("  PASS: Seeded runs match identically.")

Source 1 (Before fix - as_completed):
  Run 1 finish order: ['yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-03.parquet', 'yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-04.parquet'] | first 5 fares: [7.2, 11.4, 44.3, 12.8, 19.8]
  Run 2 finish order: ['yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-03.parquet', 'yellow_tripdata_2026-04.parquet'] | first 5 fares: [7.2, 11.4, 44.3, 12.8, 19.8]

Source 1 (After fix - pool.map):
  Run 1 order: ['yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-03.parquet', 'yellow_tripdata_2026-04.parquet'] | first 5 fares: [7.2, 7.9, 10.7, 38.7, 13.5]
  Run 2 order: ['yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-03.parquet', 'yellow_tripdata_2026-04.parquet'] | first 5 fares: [7.2, 7.9, 10.7, 38.7, 13.5]
  PASS: Fixed shard-order runs match identically.

Source 2 (Before fix - unseeded SGD): Run 1 F1=0.9332

Cell 7 — Total Runtime

In [3]:
print(f"Total notebook runtime: {time.perf_counter() - START_TIME:.2f} seconds")

NameError: name 'time' is not defined